# 4 · A zoo of finite element spaces

In unit 3 a **CoefficientFunction** was a function of a *mapped integration point*.
Here we make the **basis (shape) functions** systematic and tie them to the **mesh**.

- A **finite element space** (`FESpace`) *is* the collection of those basis functions.
- Every FE function is a **linear combination** of them (cf. unit 3).
- The coefficients live in a **`GridFunction`**, which is *itself* a CoefficientFunction
  (give it a mapped point, get a value).

So we can finally *look at* the basis: set one coefficient to `1`, the rest to `0`, and draw.

In [ ]:
# --- Google Colab: install NGSolve on first run (a no-op anywhere else) -------
# NGSolve ships its PyPI wheels as pre-releases, so the `--pre` flag is essential.
import sys
if "google.colab" in sys.modules:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "--pre",
                    "ngsolve", "anywidget"], check=True)

In [ ]:
from ngsolve import *
from ngsolve.webgui import Draw
from netgen.occ import *              # WorkPlane / Glue — for the named-region mesh in section 6

mesh = Mesh(unit_square.GenerateMesh(maxh=0.4))

A `GridFunction` lives in a `FESpace` and **is** a CoefficientFunction — it evaluates at a
mapped point ... : 

  `gfu`$(x) = \sum_i $ `gfu.vec[i]` $\cdot \varphi_i(x)$  $\leadsto$ the `FESpace` provides $\{ \varphi_i(x) \}$.

In [ ]:
gf_demo = GridFunction(H1(mesh, order=2))
gf_demo.vec[13] = 3.0
print([gf_demo.vec[i] for i in range(len(gf_demo.vec))])
print("a GridFunction is a CoefficientFunction:  gf(0.3, 0.4) =", gf_demo(mesh(0.3, 0.4)))

In [ ]:
def shape_functions(space, dofs):
    """A multidim GridFunction holding one basis function per requested dof."""
    gf = GridFunction(space, multidim=len(dofs))
    for i, d in enumerate(dofs):
        gf.vecs[i][:] = 0
        gf.vecs[i][d] = 1                  # activate a single basis function
    return gf

## 1. `H1` — continuous shape functions

`H1` (Lagrange) shape functions are **continuous** across element edges — their graphs
join up with no jumps.

That continuity makes `H1` the right home for second-order problems like Poisson and heat.
Press play to flip through a few.

In [ ]:
fesH1 = H1(mesh, order=2)
gf = shape_functions(fesH1, [2, 10, 15])
Draw(gf, mesh, "H1 shape fn", interpolate_multidim=False, animate=True, deformation=True)

## 2. `L2` — discontinuous shape functions

`L2` functions live **independently on each element** — neighbours share nothing, so a
shape function is a bump confined to one triangle.

- That locality is what the discontinuous-Galerkin transport scheme exploits (notebook 12).
- It also gives the **block-diagonal mass matrix**.

In [ ]:
fesL2 = L2(mesh, order=3)
gf = shape_functions(fesL2, [0, 7, 12, 14, 25])
Draw(gf, mesh, "L2 shape fn", interpolate_multidim=False, animate=True, deformation=True)

## 3. `VectorH1` — fully continuous vector fields

`VectorH1` stacks one **`H1` per coordinate**, so the *whole* vector is continuous across
edges.

- It is the natural home for **displacements** (elasticity) and **velocities** (Taylor–Hood, unit 7).
- Each shape function activates **one component** of one node: an `H1` bump in $x$ *or* $y$.
- Here **both** components are continuous; the next spaces keep only **one**.
- Its discontinuous twin is **`VectorL2`** (a vector `L2`, with a block-diagonal mass matrix).

In [ ]:
fesVecH1 = VectorH1(mesh, order=2)
gf = shape_functions(fesVecH1, [4, 20, 40, 55])
Draw(gf, mesh, "VectorH1 shape fn", interpolate_multidim=False, animate=True, vectors={"grid_size": 25})

## 4. `HDiv` and `HCurl` — vector-valued, partially continuous

These vector spaces keep only **one component** continuous across an edge:

- **`HDiv`** — the *normal* component is continuous (ideal for fluxes / flow).
- **`HCurl`** — the *tangential* component is continuous (ideal for electromagnetics).

We draw them as a **vector grid** of arrows: across an edge the continuous component lines up
while the other may jump.

In [ ]:
fesHDiv = HDiv(mesh, order=2)
gf = shape_functions(fesHDiv, [15, 30, 45, 55])
Draw(gf, mesh, "HDiv shape fn", interpolate_multidim=False, animate=True, vectors={"grid_size": 25})

In [ ]:
fesHCurl = HCurl(mesh, order=2)
gf = shape_functions(fesHCurl, [12, 26, 40, 52])
Draw(gf, mesh, "HCurl shape fn", interpolate_multidim=False, animate=True, vectors={"grid_size": 25})

## 5. The whole catalogue — every space, by value type

Beyond the examples above NGSolve exposes **many** more spaces, all sharing the same
`(mesh, order=…)` / `.ndof` / `.TnT()` interface.

- We ask Python for **all** of them and sort by value type — *scalar*, *vector* or *matrix*.
- The kind is just a trial function's `.dim` (1 / 2 / 4 on a 2-D mesh).
- Pick any name and drop it into `shape_functions` above to see what it is made of.

In [ ]:
# list all (easily visible) FESpace...
import ngsolve, os
from collections import defaultdict

def all_fespaces():
    """Every FESpace class reachable from the `ngsolve` namespace."""
    return sorted(n for n in dir(ngsolve) if isinstance(getattr(ngsolve, n), type)
                  and issubclass(getattr(ngsolve, n), ngsolve.FESpace) and getattr(ngsolve, n) is not ngsolve.FESpace)

def value_kind(name):
    try:
        d = getattr(ngsolve, name)(mesh).TrialFunction().dim   # 1 / 2 / 4 on a 2-D mesh
        return {1: "scalar", 2: "vector", 4: "matrix"}.get(d, f"dim {d}")
    except Exception:
        return "special"          # base-space wrappers, surface-only, …

## this is dirty python stuff ...
groups = defaultdict(list)
saved = devnull = None                                 # mute a couple of chatty C++ ctors
try:
    saved = os.dup(1); devnull = os.open(os.devnull, os.O_WRONLY); os.dup2(devnull, 1)
except Exception:
    saved = None
try:
    for name in all_fespaces():
        groups[value_kind(name)].append(name)
finally:
    if saved is not None:
        os.dup2(saved, 1); os.close(saved); os.close(devnull)

for kind in ["scalar", "vector", "matrix", "special"]:
    print(f"{kind:>7}:  {', '.join(groups[kind])}\n--------")

In [ ]:
# Example:
fes_test = HCT_FESpace(mesh)
gf = shape_functions(fes_test, [0,10,50])
Draw(gf, mesh, "test shape fn", interpolate_multidim=False, animate=True, deformation=True)

## 6. Setting values on regions and boundaries

`Set` and `Interpolate` project a CoefficientFunction into a `GridFunction`.

- **`definedon=`** restricts that to a **named material** (`mesh.Materials(...)`) or a
  **boundary** (`mesh.Boundaries(...)`).
- Use it for different fields on different subdomains, or to write **boundary data** (how
  Dirichlet values are lifted, notebook 5).
- Each `Set`/`Interpolate` **zeros the whole vector first**, then writes its region — so to
  fill several regions at once, pass a region-wise `MaterialCF`/`BoundaryCF` to a *single* `Set`.

(More: i-tutorial 1.5, *spaces & forms on subdomains*.)

In [ ]:
# a two-material strip with named regions ("left"/"right") and boundaries ("top"/"bot")
left  = WorkPlane().Rectangle(1, 1).Face();               left.name  = "left"
right = WorkPlane().MoveTo(1, 0).Rectangle(1, 1).Face();  right.name = "right"
strip = Glue([left, right]); strip.edges.Min(Y).name = "bot"; strip.edges.Max(Y).name = "top"
m2 = Mesh(OCCGeometry(strip, dim=2).GenerateMesh(maxh=0.12))
print("materials:", m2.GetMaterials(), " · boundaries:", sorted(set(m2.GetBoundaries())))

# Interpolate a field onto one MATERIAL (definedon=Materials; the rest stays 0):
gf = GridFunction(H1(m2, order=3))
gf.Interpolate(0.25 + 0.25*sin(7 * x), definedon=m2.Materials("right"))
Draw(gf, m2, "Interpolate on the 'right' material", deformation=True)

In [ ]:
# Set BOUNDARY data on a named edge (definedon=Boundaries) — how Dirichlet data is lifted:
gb = GridFunction(H1(m2, order=3))
gb.Set(sin(8 * x), definedon=m2.Boundaries("top"))
Draw(gb, m2, "Set boundary data on the 'top' edge")

> **Careful:**
>
> Each `Set` or `Interpolate` first empties your `GridFunction`s vector.

In [ ]:
gb.Set(sin(8 * x), definedon=m2.Boundaries("bot"))
Draw(gb, m2, "Set boundary data on the 'bot' edge")

## Supplementary — how dofs are classified

*(Supplementary.)* NGSolve tags every dof with a **`COUPLING_TYPE`** — how it couples between
elements (just what the `bddc` / static-condensation solvers of notebook 6 exploit).

The full vocabulary:

* **`LOCAL_DOF`** — interior to a single element (condensable);
* **`INTERFACE_DOF`** — shared on an edge/face between elements;
* **`WIREBASKET_DOF`** — the coarse skeleton (vertices, …) `bddc` builds its coarse space on;
* **`UNUSED_DOF`** — allocated but unused (e.g. after refinement, or an inactive region);
* **`HIDDEN_DOF`** — element-local *and* eliminated — never enters the global system
  (hidden dofs; exploited heavily by the
  [HDG tricks](https://docu.ngsolve.org/ngs24/CFD/hdg_tricks.html) of the NGSolve User Meeting 2024);
* plus the composite tags **`EXTERNAL`**, **`NONWIREBASKET`**, **`CONDENSABLE`**, **`VISIBLE`**,
  **`ANY`** — unions of the above, handy when querying free dofs / condensation.

`Compress(V)` drops the `UNUSED`/`HIDDEN` ones from the global numbering. Just ask the space:

In [ ]:
from collections import Counter
fes = H1(mesh, order=3)
kinds = Counter(str(fes.CouplingType(i)).split(".")[-1] for i in range(fes.ndof))
print(f"H1 order 3 — {fes.ndof} dofs: {dict(kinds)}")

Next: **solving** — how the linear system is actually handled
(free dofs, lifting boundary data, static condensation).

In [ ]:
# Navigation between units — shown only in a live notebook (Colab / JupyterLite /
# local Jupyter), never in the rendered website (which has its own prev/next nav).
import os, sys
if not os.environ.get("WEBGUI_SCENE_DIR"):          # not the static site build
    _prev = ("03-coefficientfunctions", "3 · What is a CoefficientFunction?")
    _next = ("05-solving", "5 · First linear solve")
    def _u(_nb):
        if "google.colab" in sys.modules:
            return "https://colab.research.google.com/github/schruste/ngsum2026-colab/blob/colab/" + _nb + ".ipynb"
        return _nb + ".ipynb"                       # JupyterLite & local: relative .ipynb link
    _parts  = ["⬅️ **Previous:** [%s](%s)" % (_prev[1], _u(_prev[0]))] if _prev else []
    _parts += ["➡️ **Next:** [%s](%s)" % (_next[1], _u(_next[0]))] if _next else []
    from IPython.display import display, Markdown
    display(Markdown(" · ".join(_parts)))